## 1. Raw Data Acquisition

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("so2-pipeline-m2") \
    .getOrCreate()

so2_sample = spark.read.parquet(
    "gs://st446-m2-outputs/tropomi/SO2/2023-06-01/*.parquet"
)
no2_sample = spark.read.parquet(
    "gs://st446-m2-outputs/tropomi/NO2/2023-06-01/*.parquet"
)
era5_sample = spark.read.parquet(
    "gs://st446-m2-outputs/era5/wind_850hpa/2023-06-01.parquet"
)

so2_sample.printSchema()
no2_sample.printSchema()
era5_sample.printSchema()

so2_expected = ['lat','lon','time','date','orbit','vcd','qa_value',
                'cloud_fraction','solar_zenith_angle','sensor_zenith_angle',
                'so2_vcd_1km','so2_vcd_7km','aerosol_index','surface_altitude']
era5_expected = ['date','hour','era5_lat','era5_lon','u_wind_850','v_wind_850']

for col in so2_expected:
    assert col in so2_sample.columns, f"Missing SO2 column: {col}"
for col in era5_expected:
    assert col in era5_sample.columns, f"Missing ERA5 column: {col}"

print("All schema checks passed")

26/04/29 20:00:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


root
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- time: timestamp_ntz (nullable = true)
 |-- date: date (nullable = true)
 |-- orbit: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- vcd: float (nullable = true)
 |-- qa_value: float (nullable = true)
 |-- cloud_fraction: float (nullable = true)
 |-- solar_zenith_angle: float (nullable = true)
 |-- sensor_zenith_angle: float (nullable = true)
 |-- so2_vcd_1km: float (nullable = true)
 |-- so2_vcd_7km: float (nullable = true)
 |-- aerosol_index: float (nullable = true)
 |-- surface_altitude: float (nullable = true)

root
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- time: timestamp_ntz (nullable = true)
 |-- date: date (nullable = true)
 |-- orbit: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- vcd: float (nullable = true)
 |-- qa_value: float (nullable = true)
 |-- cloud_fraction: float (nullable = true)
 |-- solar_zenith_angle: float (nul

## 2. Format Conversion & Storage

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("so2-pipeline-m2") \
    .getOrCreate()

so2_sample = spark.read.parquet(
    "gs://st446-m2-outputs/tropomi/SO2/2023-06-01/*.parquet"
)
no2_sample = spark.read.parquet(
    "gs://st446-m2-outputs/tropomi/NO2/2023-06-01/*.parquet"
)
era5_sample = spark.read.parquet(
    "gs://st446-m2-outputs/era5/wind_850hpa/2023-06-01.parquet"
)

so2_sample.printSchema()
no2_sample.printSchema()
era5_sample.printSchema()

so2_expected = ['lat','lon','time','date','orbit','vcd','qa_value',
                'cloud_fraction','solar_zenith_angle','sensor_zenith_angle',
                'so2_vcd_1km','so2_vcd_7km','aerosol_index','surface_altitude']
era5_expected = ['date','hour','era5_lat','era5_lon','u_wind_850','v_wind_850']

for col in so2_expected:
    assert col in so2_sample.columns, f"Missing SO2 column: {col}"
for col in era5_expected:
    assert col in era5_sample.columns, f"Missing ERA5 column: {col}"

print("All schema checks passed")

root
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- time: timestamp_ntz (nullable = true)
 |-- date: date (nullable = true)
 |-- orbit: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- vcd: float (nullable = true)
 |-- qa_value: float (nullable = true)
 |-- cloud_fraction: float (nullable = true)
 |-- solar_zenith_angle: float (nullable = true)
 |-- sensor_zenith_angle: float (nullable = true)
 |-- so2_vcd_1km: float (nullable = true)
 |-- so2_vcd_7km: float (nullable = true)
 |-- aerosol_index: float (nullable = true)
 |-- surface_altitude: float (nullable = true)

root
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- time: timestamp_ntz (nullable = true)
 |-- date: date (nullable = true)
 |-- orbit: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- vcd: float (nullable = true)
 |-- qa_value: float (nullable = true)
 |-- cloud_fraction: float (nullable = true)
 |-- solar_zenith_angle: float (nul

## 3. Quality Assurance

In [3]:
so2 = spark.read.parquet("gs://st446-m2-outputs/tropomi/SO2/2023-06-0[12345]/*.parquet")
no2 = spark.read.parquet("gs://st446-m2-outputs/tropomi/NO2/2023-06-0[12345]/*.parquet")
era5 = spark.read.parquet("gs://st446-m2-outputs/era5/wind_850hpa/2023-06-0[12345].parquet")

print(f"SO2 rows: {so2.count():,}")
print(f"NO2 rows: {no2.count():,}")
print(f"ERA5 rows: {era5.count():,}")

SO2 rows: 46,812,769


NO2 rows: 41,127,863
ERA5 rows: 124,588,800


## 4. SO₂ / NO₂ Spatial Co-location

In [ ]:
import math

# Increase shuffle partitions for full month dataset
spark.conf.set("spark.sql.shuffle.partitions", "400") #adjust depending on dataset size

# Snap SO2 to 0.1 degree grid
so2_snapped = so2.withColumn(
    "lat_key",
    (F.round(F.col("lat") / 0.1) * 0.1).cast("float")
).withColumn(
    "lon_key",
    (F.round(F.col("lon") / 0.1) * 0.1).cast("float")
)

# Snap NO2 to same 0.1 degree grid and select only needed columns
no2_snapped = no2.withColumn(
    "lat_key",
    (F.round(F.col("lat") / 0.1) * 0.1).cast("float")
).withColumn(
    "lon_key",
    (F.round(F.col("lon") / 0.1) * 0.1).cast("float")
).select(
    "date",
    "lat_key",
    "lon_key",
    F.col("vcd").alias("no2_vcd")
)

# Simple equi-join on date + snapped grid
coloc = so2_snapped.join(
    no2_snapped,
    on=["date", "lat_key", "lon_key"],
    how="left"
).drop("lat_key", "lon_key")

# Checkpoint to GCS
coloc.write.mode("overwrite").parquet(
    "gs://st446-m2-outputs/intermediate/coloc_june2023_nowindcorrection.parquet"
)

print("Co-location done, reading back checkpoint...")
coloc = spark.read.parquet(
    "gs://st446-m2-outputs/intermediate/coloc_june2023_nowindcorrection.parquet"
)
print(f"Co-located pixel pairs: {coloc.count():,}")

In [6]:
#checkpoint script for re-runs (run step 4 first on any new training sets)
coloc = spark.read.parquet(
    "gs://st446-m2-outputs/intermediate/coloc_june2023_nowindcorrection.parquet"
)
print(f"Co-located pixel pairs: {coloc.count():,}")

Co-located pixel pairs: 160,275,037


## 5. ERA5 Wind Join

In [7]:
coloc = coloc.withColumn(
    "tropomi_lon_360",
    ((F.col("lon") + 360) % 360)
)

coloc = coloc \
    .withColumn("era5_lat_key",
        (F.round(F.col("lat") / 0.25) * 0.25).cast("float")
    ) \
    .withColumn("era5_lon_key",
        (F.round(F.col("tropomi_lon_360") / 0.25) * 0.25).cast("float")
    ) \
    .withColumn("hour", F.hour(F.col("time")))

era5_keyed = era5 \
    .withColumn("era5_lat_key", F.col("era5_lat").cast("float")) \
    .withColumn("era5_lon_key", F.col("era5_lon").cast("float"))

coloc = coloc.join(
    era5_keyed.select(
        "date", "hour", "era5_lat_key", "era5_lon_key",
        "u_wind_850", "v_wind_850"
    ),
    on=["date", "hour", "era5_lat_key", "era5_lon_key"],
    how="left"
)

coloc = coloc.withColumn(
    "wind_speed",
    F.sqrt(F.col("u_wind_850")**2 + F.col("v_wind_850")**2)
)

print(f"Rows after ERA5 join: {coloc.count():,}")
print(f"Rows missing ERA5 wind: {coloc.filter('u_wind_850 is null').count():,}")

Rows after ERA5 join: 160,275,037


Rows missing ERA5 wind: 66,937


## 6. Wind Back-Trajectory Correction

In [10]:
R_EARTH_M = 6371000.0
SO2_RESIDENCE_SECONDS = 86400
DEG_PER_METER_LAT = 1.0 / (R_EARTH_M * math.pi / 180.0)

coloc = coloc \
    .withColumn(
        "source_lat",
        F.col("lat") - (
            F.col("v_wind_850") * SO2_RESIDENCE_SECONDS * DEG_PER_METER_LAT
        )
    ) \
    .withColumn(
        "source_lon",
        F.col("lon") - (
            F.col("u_wind_850") * SO2_RESIDENCE_SECONDS * DEG_PER_METER_LAT /
            F.cos(F.col("lat") * math.pi / 180.0)
        )
    ) \
    .withColumn(
        "source_lat",
        F.greatest(F.lit(-90.0), F.least(F.lit(90.0), F.col("source_lat")))
    )

## 7. Fioletov Catalogue Broadcast Spatial Join (raw pixel distance)

In [11]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, FloatType, StringType, IntegerType

fioletov_pd = pd.read_csv("gs://st446-m2-outputs/fioletov_catalogue.csv")
fioletov_bc = spark.sparkContext.broadcast(fioletov_pd)

MAX_LABEL_DIST_KM = 150.0

result_schema = StructType([
    StructField("nearest_source_name", StringType()),
    StructField("source_type", StringType()),
    StructField("country", StringType()),
    StructField("raw_pixel_dist_km", FloatType()),  # renamed for ablation
    StructField("source_elevation_m", IntegerType()),
    StructField("source_amf", FloatType()),
])

@pandas_udf(result_schema)
def nearest_source_udf(source_lats: pd.Series, source_lons: pd.Series) -> pd.DataFrame:
    cat = fioletov_bc.value
    cat_lats = np.radians(cat["latitude"].values)
    cat_lons = np.radians(cat["longitude"].values)
    
    # Vectorised haversine — processes all pixels simultaneously
    slats = np.radians(source_lats.values)[:, np.newaxis]  # (N, 1)
    slons = np.radians(source_lons.values)[:, np.newaxis]  # (N, 1)
    
    dlat = cat_lats - slats  # (N, 759)
    dlon = cat_lons - slons  # (N, 759)
    
    a = (np.sin(dlat/2)**2 +
         np.cos(slats) * np.cos(cat_lats) * np.sin(dlon/2)**2)
    dist_km = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))  # (N, 759)
    
    idx = np.argmin(dist_km, axis=1)  # (N,)
    min_dists = dist_km[np.arange(len(idx)), idx]  # (N,)
    
    return pd.DataFrame({
            "nearest_source_name": cat["source_name"].iloc[idx].values,
            "source_type": cat["source_type"].iloc[idx].values,
            "country": cat["country"].iloc[idx].values,
            "raw_pixel_dist_km": min_dists.astype(np.float32),  # renamed
            "source_elevation_m": cat["elevation_m"].iloc[idx].values.astype(np.int32),
            "source_amf": cat["amf"].iloc[idx].values.astype(np.float32),
        })

coloc = coloc.withColumn(
    "nearest",
    nearest_source_udf(F.col("lat"), F.col("lon"))  
).select("*", "nearest.*").drop("nearest")

coloc = coloc.filter(F.col("raw_pixel_dist_km") <= MAX_LABEL_DIST_KM)

coloc = coloc.withColumn(
    "label",
    F.when(F.col("source_type") == "Volcano", 1).otherwise(0)
)

print(f"Labelled pixels: {coloc.count():,}")
print(f"Volcanic (label=1): {coloc.filter('label=1').count():,}")
print(f"Industrial (label=0): {coloc.filter('label=0').count():,}")

Labelled pixels: 12,903,115


Volcanic (label=1): 1,521,213


Industrial (label=0): 11,381,902


## 8. Feature Engineering

In [12]:
coloc = coloc \
    .withColumn(
        "so2_no2_ratio",
        F.col("vcd") / F.when(
            F.col("no2_vcd") != 0, F.col("no2_vcd")
        ).otherwise(None)
    ) \
    .withColumn(
        "vcd_height_ratio",
        F.col("so2_vcd_7km") / F.when(
            F.col("so2_vcd_1km") != 0, F.col("so2_vcd_1km")
        ).otherwise(None)
    ) \
    .withColumn(
        "raw_dist_km",
        F.sqrt(
            ((F.col("lat") - F.col("source_lat")) * 111.0)**2 +
            ((F.col("lon") - F.col("source_lon")) * 111.0 *
             F.cos(F.col("lat") * math.pi / 180.0))**2
        )
    ) \
    .withColumn(
        "wind_x_ratio",
        F.col("wind_speed") * F.col("so2_no2_ratio")
    ) \
    .withColumn("month", F.month(F.col("time"))) \
    .withColumn("hour_of_day", F.hour(F.col("time")))

feature_cols = [
    "nearest_source_name",        
    "so2_no2_ratio",
    "vcd_height_ratio",
    "aerosol_index",
    "raw_pixel_dist_km",          # ablation version — raw pixel distance
    "raw_dist_km",
    "source_elevation_m",
    "wind_speed",
    "wind_x_ratio",
    "cloud_fraction",
    "solar_zenith_angle",
    "sensor_zenith_angle",
    "surface_altitude",
    "month",
    "hour_of_day",
    "label"
]

final = coloc.select(feature_cols).dropna(subset=[
    "so2_no2_ratio", "vcd_height_ratio", "raw_pixel_dist_km", "label"
])

print(f"Final feature rows: {final.count():,}")
final.describe().show()

Final feature rows: 12,708,842


26/04/29 20:56:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+--------------------+-------------------+--------------------+-----------------+------------------+------------------+-------------------+--------------------+-------------------+------------------+-------------------+------------------+--------+-----------------+-------------------+
|summary|nearest_source_name|       so2_no2_ratio|   vcd_height_ratio|       aerosol_index|raw_pixel_dist_km|       raw_dist_km|source_elevation_m|         wind_speed|        wind_x_ratio|     cloud_fraction|solar_zenith_angle|sensor_zenith_angle|  surface_altitude|   month|      hour_of_day|              label|
+-------+-------------------+--------------------+-------------------+--------------------+-----------------+------------------+------------------+-------------------+--------------------+-------------------+------------------+-------------------+------------------+--------+-----------------+-------------------+
|  count|           12708842|            12708842|        

In [13]:
OUTPUT_PATH = "gs://st446-m2-outputs/features/pilot_june2023_1_5day_nowindcorrection.parquet"

final.write \
    .mode("overwrite") \
    .parquet(OUTPUT_PATH)

verify = spark.read.parquet(OUTPUT_PATH)
print(f"Written rows: {verify.count():,}")
print(f"Columns: {verify.columns}")
print(f"Volcanic pixels: {verify.filter('label=1').count():,}")
print(f"Industrial pixels: {verify.filter('label=0').count():,}")

Written rows: 12,708,842
Columns: ['nearest_source_name', 'so2_no2_ratio', 'vcd_height_ratio', 'aerosol_index', 'raw_pixel_dist_km', 'raw_dist_km', 'source_elevation_m', 'wind_speed', 'wind_x_ratio', 'cloud_fraction', 'solar_zenith_angle', 'sensor_zenith_angle', 'surface_altitude', 'month', 'hour_of_day', 'label']


Volcanic pixels: 1,499,449


Industrial pixels: 11,209,393
